# Environment Preparation

## Install the `ifran` library

In [2]:
! pip install --upgrade --force-reinstall "ifran @ git+https://github.com/ifran-project/ifran.git@develop"

  Cloning https://github.com/ifran-project/ifran.git (to revision develop) to /tmp/pip-install-8t1vu2ij/ifran_32159d8591564ab19d6640e444a2cc7f
  Running command git clone --filter=blob:none --quiet https://github.com/ifran-project/ifran.git /tmp/pip-install-8t1vu2ij/ifran_32159d8591564ab19d6640e444a2cc7f
  Running command git checkout -b develop --track origin/develop
  Switched to a new branch 'develop'
  branch 'develop' set up to track 'origin/develop'.
  Resolved https://github.com/ifran-project/ifran.git to commit b3a5d7323d12aa553624350b8635e21e5b1e4c1e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ansible_core-2.21.4-py3-none-any.whl.metadata (7.8 kB)
  Using cached ansible_runner-2.4.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached metamodel_3gpp-0.0.6-py3-none-any.whl.metadata (5.2 kB)
  Using cached pydantic-2.13.5-py3-none-any.whl.metadata (110 kB)
  Using cached pydant

In some setup, it might be necessary to install ansible galaxy dependencies as follows.

In [3]:
! ansible-galaxy collection install -r ../../ansible/requirements.yaml

Starting galaxy collection install process
Process install dependency map
Starting collection install process
Installing 'ansible.posix:2.2.2' to '/home/vscode/.ansible/collections/ansible_collections/ansible/posix'
ansible.posix:2.2.2 was installed successfully
Installing 'community.general:14.0.1' to '/home/vscode/.ansible/collections/ansible_collections/community/general'
community.general:14.0.1 was installed successfully
Installing 'kubernetes.core:6.6.0' to '/home/vscode/.ansible/collections/ansible_collections/kubernetes/core'
kubernetes.core:6.6.0 was installed successfully
Installing 'ansible.mysql:5.2.1' to '/home/vscode/.ansible/collections/ansible_collections/ansible/mysql'
ansible.mysql:5.2.1 was installed successfully
Installing 'community.mongodb:1.8.1' to '/home/vscode/.ansible/collections/ansible_collections/community/mongodb'
community.mongodb:1.8.1 was installed successfully
Installing 'community.library_inventory_filtering_v1:1.1.5' to '/home/vscode/.ansible/collect

The `ifran` orchestrator is not tied to a specific testbed. It can be used on any infrastructure, as long as the machines:
  - run **Linux**,
  - are reachable over **SSH**, and
  - can run **Kubernetes** (the 5G functions are deployed on a Kubernetes cluster).

In that case, declare the machines with `provisioner: null`, mark them `is_provisioned: true` and give their `ssh_info`. The orchestrator then skips provisioning and goes straight to building the Kubernetes cluster and deploying the network. If you already have a Kubernetes cluster, you can also use it directly, as long as its nodes carry the expected role labels.

The `ifran` orchestrator is also built to integrate seamlessly with the [SLICES-RI](https://doc.slices-ri.eu/) infrastructure, with built-in provisioners (e.g., `bi`, `pos`, `r2lab`) to reserve and set up VMs, bare-metal nodes, radio units and phones for you, so a complete network, including real radio equipment, can be deployed from a single description.

## Integration with SLICES-RI

### Prerequisites

To use the `ifran` orchestrator with SLICES-RI, ensure that you have:
* A valid SLICES account
* A basic understanding of the SLICES CLI

### Step 1 – Register for SLICES

In SLICES, resources are structured into **projects** and **experiments**. Every experiment belongs to a specific project. 

Make sure you have an active SLICES-RI account and are registered with a project. Visit the [SLICES Portal](https://portal.slices-ri.eu/) for details on account creation and project management.

### Step 2 – Install and Configure the SLICES CLI

1. Install the [SLICES Command Line Interface (SLICES CLI)](https://doc.slices-ri.eu/SupportingServices/slicescli.html).
2. Log in using your SLICES credentials.
3. Set your active project context (for example, `post5g-beta`).
4. Install the POS subcommand for the SLICES CLI.

In [4]:
! pip install --upgrade slices-cli --extra-index-url=https://doc.slices-ri.eu/pypi/

Looking in indexes: https://pypi.org/simple, https://doc.slices-ri.eu/pypi/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [slices-cli]5 [slices-cli]ai]e]-aiis]


In [5]:
! slices auth login

Please go to 
https://portal.slices-ri.eu/oauth/authorize_device?user_code=DHQX-VSQL and login
to continue.
⠋ Waiting for authorizationon
Successfully logged in as 'dsaucezi' 
💾 Saved to /home/vscode/.slices/auth.json
The following projects are available: cnria2026, polars-benchmark, post5g-beta, 
uclan-cy-playground-1, uoulu-5gtn-ri
👉 Now select the project that you want to use with slices project use <project 
name>


Select your project with the following command (adapt the project name to yours).

In [6]:
! slices project use post5g-beta

The current project was set to 'post5g-beta', in which you are a lead and which 
expires on 2026-12-31 23:59 UTC.


If you haven't already done so, [register your public SSH key](https://doc.slices-ri.eu/SupportingServices/slicescli.html#registering-your-public-ssh-key) with SLICES. For example by running:

```bash
ssh-keygen -t ed25519 -N "" -f ~/.ssh/id_ed25519
slices pubkey register ~/.ssh/id_ed25519.pub 
```

These keys are automatically provisioned on all infrastructure resources utilized throughout the `ifran` library.

> **Note:** This initialization and configuration process is required only once when setting up the SLICES CLI.

If you plan on using bare-metal machines (`pos` provisioner), install the POS submodule for the SLICES-CLI:

In [7]:
! pip install --upgrade slices-cli-pos --index-url https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple --extra-index-url https://doc.slices-ri.eu/pypi/

Looking in indexes: https://gitlab.inria.fr/api/v4/projects/68732/packages/pypi/simple, https://doc.slices-ri.eu/pypi/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 8.6 MB/s eta 0:00:00a 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.4/587.4 kB 9.3 MB/s eta 0:00:00
  Attempting uninstall: requests━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/21 [setuptools]
    Found existing installation: requests 2.34.2━━━━━━━━━━━━━━  2/21 [setuptools]
    Uninstalling requests-2.34.2:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  2/21 [setuptools]
      Successfully uninstalled requests-2.34.2━━━━━━━━━━━━━━━━  2/21 [setuptools]
  Attempting uninstall: python-dateutil━━━━━━━━━━━━━━━━━━━━━━━━━━━  3/21 [requests]
    Found existing installation: python-dateutil 4.0.0━━━━━━━━  3/21 [requests]
    Uninstalling python-dateutil-4.0.0:━━━━━━━━━━━━━━━━━━━━━━━  3

You are now fully prepared to run your experiment!